# Customer Segmentation with RFM Analysis & K-Means Clustering

**Date:** September 2026

---

## Business Context

An e-commerce company wants to **segment its customer base** to design targeted marketing campaigns, optimize retention strategies, and maximize customer lifetime value (CLV).

We will use **RFM Analysis** (Recency, Frequency, Monetary) combined with **K-Means Clustering** to identify distinct customer segments from transactional data.

### Approach
1. Generate transactional data (~5,000 customers, ~50,000 transactions)
2. Perform exploratory data analysis (EDA)
3. Calculate RFM metrics and assign quintile scores
4. Apply K-Means clustering on standardized RFM features
5. Profile each cluster and derive actionable business recommendations

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime, timedelta

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficos
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# Reproducibilidad
SEED = 42
np.random.seed(SEED)

print('All imports loaded successfully.')

## 2. Data Loading & Preparation

We create realistic transactional data with different customer behavioral archetypes:
- **VIP customers** (~5%): high frequency, high spend
- **Regular customers** (~25%): moderate activity
- **Occasional buyers** (~40%): low-to-mid frequency
- **Dormant/churned** (~30%): haven't purchased recently

In [ ]:
# Parámetros de generación
N_CUSTOMERS = 5000
REFERENCE_DATE = datetime(2026, 9, 30)  # Fecha de análisis
START_DATE = REFERENCE_DATE - timedelta(days=730)  # 2 años atrás

CATEGORIES = ['Electronics', 'Clothing', 'Home & Garden', 'Books', 
              'Sports', 'Beauty', 'Food & Grocery', 'Toys']

# Definir arquetipos de clientes
archetypes = {
    'vip':        {'pct': 0.05, 'freq_range': (30, 80), 'amount_mean': 150, 'amount_std': 60, 'recency_max': 30},
    'regular':    {'pct': 0.25, 'freq_range': (10, 30), 'amount_mean': 75,  'amount_std': 35, 'recency_max': 90},
    'occasional': {'pct': 0.40, 'freq_range': (3, 10),  'amount_mean': 50,  'amount_std': 25, 'recency_max': 200},
    'dormant':    {'pct': 0.30, 'freq_range': (1, 5),   'amount_mean': 40,  'amount_std': 20, 'recency_max': 730},
}

transactions = []
customer_id_counter = 1

for archetype, params in archetypes.items():
    n_cust = int(N_CUSTOMERS * params['pct'])
    
    for _ in range(n_cust):
        cid = f'C{customer_id_counter:05d}'
        customer_id_counter += 1
        
        # Número de transacciones para este cliente
        n_txns = np.random.randint(params['freq_range'][0], params['freq_range'][1] + 1)
        
        # Generar fechas de transacción
        if archetype == 'dormant':
            # Clientes dormidos: transacciones concentradas hace tiempo
            days_ago = np.random.randint(180, params['recency_max'], size=n_txns)
        else:
            days_ago = np.random.randint(0, params['recency_max'], size=n_txns)
        
        dates = [REFERENCE_DATE - timedelta(days=int(d)) for d in days_ago]
        
        # Generar montos (lognormal para simular distribución realista)
        amounts = np.random.lognormal(
            mean=np.log(params['amount_mean']), 
            sigma=params['amount_std'] / params['amount_mean'],
            size=n_txns
        )
        amounts = np.clip(amounts, 5, 2000)  # Limitar rango
        
        # Preferencias de categoría por cliente
        cat_probs = np.random.dirichlet(np.ones(len(CATEGORIES)) * 0.5)
        categories = np.random.choice(CATEGORIES, size=n_txns, p=cat_probs)
        
        for date, amount, cat in zip(dates, amounts, categories):
            transactions.append({
                'customer_id': cid,
                'transaction_date': date,
                'amount': round(float(amount), 2),
                'product_category': cat
            })

# Crear DataFrame
df = pd.DataFrame(transactions)
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df = df.sort_values('transaction_date').reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'Unique customers: {df["customer_id"].nunique():,}')
print(f'Date range: {df["transaction_date"].min().date()} to {df["transaction_date"].max().date()}')
print(f'Total revenue: ${df["amount"].sum():,.2f}')
df.head(10)

In [ ]:
# Verificación rápida de calidad de datos
print('--- Data Quality Check ---')
print(f'\nNull values:\n{df.isnull().sum()}')
print(f'\nData types:\n{df.dtypes}')
print(f'\nBasic statistics:')
df.describe()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# --- Transactions per customer distribution ---
txns_per_customer = df.groupby('customer_id').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de frecuencia de compra
axes[0].hist(txns_per_customer, bins=50, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Number of Transactions')
axes[0].set_ylabel('Number of Customers')
axes[0].set_title('Distribution of Purchase Frequency per Customer')
axes[0].axvline(txns_per_customer.median(), color='red', linestyle='--', 
                label=f'Median: {txns_per_customer.median():.0f}')
axes[0].legend()

# Distribución de montos
axes[1].hist(df['amount'], bins=60, color='#4CAF50', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Transaction Amount ($)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Transaction Amounts')
axes[1].axvline(df['amount'].median(), color='red', linestyle='--',
                label=f'Median: ${df["amount"].median():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Monthly revenue trend ---
df['year_month'] = df['transaction_date'].dt.to_period('M')
monthly_revenue = df.groupby('year_month')['amount'].sum()

fig, ax = plt.subplots(figsize=(14, 5))
monthly_revenue.plot(kind='bar', ax=ax, color='#FF7043', edgecolor='white', width=0.8)
ax.set_title('Monthly Revenue Over Time', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Mostrar solo cada 3er label para legibilidad
tick_labels = [label.get_text() if i % 3 == 0 else '' 
               for i, label in enumerate(ax.get_xticklabels())]
ax.set_xticklabels(tick_labels, rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# --- Top 20 customers by total spend ---
top_customers = (df.groupby('customer_id')['amount']
                 .sum()
                 .sort_values(ascending=False)
                 .head(20))

fig, ax = plt.subplots(figsize=(12, 6))
top_customers.plot(kind='barh', ax=ax, color='#AB47BC', edgecolor='white')
ax.set_xlabel('Total Spend ($)')
ax.set_ylabel('Customer ID')
ax.set_title('Top 20 Customers by Total Spend', fontsize=14)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# --- Revenue by product category ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_revenue = df.groupby('product_category')['amount'].sum().sort_values(ascending=True)
cat_count = df.groupby('product_category').size().sort_values(ascending=True)

cat_revenue.plot(kind='barh', ax=axes[0], color='#26A69A', edgecolor='white')
axes[0].set_title('Revenue by Product Category')
axes[0].set_xlabel('Revenue ($)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

cat_count.plot(kind='barh', ax=axes[1], color='#42A5F5', edgecolor='white')
axes[1].set_title('Transaction Count by Product Category')
axes[1].set_xlabel('Number of Transactions')

plt.tight_layout()
plt.show()

## 4. RFM Analysis

**RFM** stands for:
- **Recency (R):** Days since last purchase
- **Frequency (F):** Total number of transactions
- **Monetary (M):** Total amount spent

In [ ]:
# --- Calcular métricas RFM ---
rfm = df.groupby('customer_id').agg(
    recency=('transaction_date', lambda x: (REFERENCE_DATE - x.max()).days),
    frequency=('transaction_date', 'count'),
    monetary=('amount', 'sum')
).reset_index()

print(f'RFM table shape: {rfm.shape}')
rfm.describe().round(2)

In [ ]:
# --- RFM distributions ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['recency', 'frequency', 'monetary']
colors = ['#EF5350', '#42A5F5', '#66BB6A']
titles = ['Recency (days since last purchase)', 'Frequency (# transactions)', 'Monetary (total spend $)']

for ax, metric, color, title in zip(axes, metrics, colors, titles):
    ax.hist(rfm[metric], bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('Count')
    ax.axvline(rfm[metric].median(), color='black', linestyle='--', alpha=0.7,
               label=f'Median: {rfm[metric].median():,.1f}')
    ax.legend(fontsize=9)

plt.suptitle('RFM Metric Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- RFM Scoring using quintiles (1-5) ---
# Para Recency: menor es mejor -> invertir scoring
rfm['R_score'] = pd.qcut(rfm['recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Composite RFM score
rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

# Segment labels basados en score compuesto
def rfm_segment_label(row):
    """Assign human-readable segment labels based on RFM scores."""
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2:
        return 'Lost'
    else:
        return 'Potential Loyalists'

rfm['rfm_segment'] = rfm.apply(rfm_segment_label, axis=1)

print('RFM Segment Distribution:')
print(rfm['rfm_segment'].value_counts())
print()
rfm.head(10)

In [ ]:
# --- Visualizar segmentos RFM ---
segment_counts = rfm['rfm_segment'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors_seg = ['#4CAF50', '#2196F3', '#FF9800', '#F44336', '#9C27B0', '#607D8B']
segment_counts.plot(kind='bar', ax=axes[0], color=colors_seg[:len(segment_counts)], edgecolor='white')
axes[0].set_title('Customer Count by RFM Segment', fontsize=13)
axes[0].set_ylabel('Number of Customers')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart
axes[1].pie(segment_counts, labels=segment_counts.index, autopct='%1.1f%%',
            colors=colors_seg[:len(segment_counts)], startangle=90, 
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('RFM Segment Proportions', fontsize=13)

plt.tight_layout()
plt.show()

## 5. K-Means Clustering

While RFM scoring uses fixed rules, **K-Means** lets the data reveal natural groupings. We standardize the RFM features and use the **Elbow Method** + **Silhouette Score** to select the optimal number of clusters.

In [ ]:
# --- Preparar datos para clustering ---
# Usamos log-transform para reducir skewness (común en datos monetarios)
rfm_features = rfm[['recency', 'frequency', 'monetary']].copy()
rfm_features['frequency'] = np.log1p(rfm_features['frequency'])
rfm_features['monetary'] = np.log1p(rfm_features['monetary'])

# Estandarizar
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)

print(f'Scaled features shape: {rfm_scaled.shape}')
print(f'Mean (should be ~0): {rfm_scaled.mean(axis=0).round(4)}')
print(f'Std  (should be ~1): {rfm_scaled.std(axis=0).round(4)}')

In [ ]:
# --- Elbow Method + Silhouette Score ---
K_RANGE = range(2, 11)
inertias = []
silhouette_scores = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10, max_iter=300)
    labels = km.fit_predict(rfm_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(rfm_scaled, labels))
    print(f'k={k:2d} | Inertia: {km.inertia_:10,.1f} | Silhouette: {silhouette_scores[-1]:.4f}')

# Gráficos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_RANGE, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method', fontsize=13)
axes[0].set_xticks(list(K_RANGE))

axes[1].plot(K_RANGE, silhouette_scores, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score by k', fontsize=13)
axes[1].set_xticks(list(K_RANGE))

# Marcar el k óptimo
best_k = list(K_RANGE)[np.argmax(silhouette_scores)]
axes[1].axvline(best_k, color='green', linestyle='--', alpha=0.7, label=f'Best k={best_k}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nOptimal k by silhouette score: {best_k}')

In [ ]:
# --- Fit final K-Means model ---
# Usamos k=4 para segmentación interpretable de negocio
OPTIMAL_K = 4

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=SEED, n_init=20, max_iter=500)
rfm['cluster'] = kmeans.fit_predict(rfm_scaled)

print(f'Silhouette Score (k={OPTIMAL_K}): {silhouette_score(rfm_scaled, rfm["cluster"]):.4f}')
print(f'\nCluster sizes:')
print(rfm['cluster'].value_counts().sort_index())

## 6. Cluster Profiling & Visualization

In [ ]:
# --- Cluster profiles (promedios de RFM por cluster) ---
cluster_profile = rfm.groupby('cluster')[['recency', 'frequency', 'monetary']].agg(
    ['mean', 'median', 'std', 'count']
).round(2)

# Tabla resumida
cluster_summary = rfm.groupby('cluster').agg(
    n_customers=('customer_id', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    median_recency=('recency', 'median'),
    median_frequency=('frequency', 'median'),
    median_monetary=('monetary', 'median'),
).round(2)

# Asignar nombres descriptivos a los clusters
# Ordenar por monetary descendente para asignar nombres
cluster_order = cluster_summary.sort_values('avg_monetary', ascending=False).index.tolist()
cluster_names = {}
name_options = ['Gold - High Value', 'Silver - Loyal', 'Bronze - Developing', 'Iron - At Risk']
for i, idx in enumerate(cluster_order):
    cluster_names[idx] = name_options[i]

rfm['cluster_name'] = rfm['cluster'].map(cluster_names)
cluster_summary['cluster_name'] = cluster_summary.index.map(cluster_names)

print('=== Cluster Summary ===')
cluster_summary

In [ ]:
# --- Box plots per cluster ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

palette = {name: color for name, color in zip(
    ['Gold - High Value', 'Silver - Loyal', 'Bronze - Developing', 'Iron - At Risk'],
    ['#FFD700', '#C0C0C0', '#CD7F32', '#708090']
)}

for ax, metric, title in zip(axes, 
                              ['recency', 'frequency', 'monetary'],
                              ['Recency (days)', 'Frequency (# txns)', 'Monetary ($)']):
    sns.boxplot(data=rfm, x='cluster_name', y=metric, ax=ax, palette=palette,
                order=['Gold - High Value', 'Silver - Loyal', 'Bronze - Developing', 'Iron - At Risk'])
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('RFM Distributions by Cluster', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3D Scatter Plot ---
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

cluster_colors = {0: '#FFD700', 1: '#C0C0C0', 2: '#CD7F32', 3: '#708090'}

for cluster_id in sorted(rfm['cluster'].unique()):
    mask = rfm['cluster'] == cluster_id
    ax.scatter(
        rfm.loc[mask, 'recency'],
        rfm.loc[mask, 'frequency'],
        rfm.loc[mask, 'monetary'],
        c=cluster_colors.get(cluster_id, '#999'),
        label=cluster_names[cluster_id],
        alpha=0.5,
        s=20,
        edgecolors='white',
        linewidth=0.3
    )

ax.set_xlabel('Recency (days)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_zlabel('Monetary ($)', fontsize=11)
ax.set_title('3D Customer Segments (RFM Space)', fontsize=14)
ax.legend(loc='upper left', fontsize=10)
ax.view_init(elev=25, azim=45)

plt.tight_layout()
plt.show()

In [ ]:
# --- Radar / Spider Chart ---
# Normalizar promedios al rango 0-1 para el radar chart
radar_data = cluster_summary[['avg_recency', 'avg_frequency', 'avg_monetary']].copy()

# Invertir recency (menor recency = mejor)
radar_data['avg_recency'] = radar_data['avg_recency'].max() - radar_data['avg_recency']

# Min-max normalize
for col in radar_data.columns:
    min_val = radar_data[col].min()
    max_val = radar_data[col].max()
    if max_val > min_val:
        radar_data[col] = (radar_data[col] - min_val) / (max_val - min_val)
    else:
        radar_data[col] = 0.5

categories_radar = ['Recency\n(lower=better)', 'Frequency', 'Monetary']
N = len(categories_radar)

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # Cerrar el polígono

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

radar_colors = ['#FFD700', '#C0C0C0', '#CD7F32', '#708090']

for i, (idx, row) in enumerate(radar_data.iterrows()):
    values = row.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=cluster_names[idx], 
            color=radar_colors[i % len(radar_colors)])
    ax.fill(angles, values, alpha=0.15, color=radar_colors[i % len(radar_colors)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories_radar, fontsize=12)
ax.set_title('Cluster Profiles - Radar Chart', fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# --- Heatmap: Cluster vs RFM Scores ---
heatmap_data = rfm.groupby('cluster_name')[['R_score', 'F_score', 'M_score']].mean()
heatmap_data = heatmap_data.reindex(['Gold - High Value', 'Silver - Loyal', 
                                      'Bronze - Developing', 'Iron - At Risk'])

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='YlOrRd', 
            linewidths=1, linecolor='white', ax=ax,
            xticklabels=['Recency Score', 'Frequency Score', 'Monetary Score'])
ax.set_title('Average RFM Scores by Cluster', fontsize=14)
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# --- Pairplot de las features RFM coloreado por cluster ---
plot_df = rfm[['recency', 'frequency', 'monetary', 'cluster_name']].copy()

g = sns.pairplot(plot_df, hue='cluster_name', palette=palette,
                 diag_kind='kde', plot_kws={'alpha': 0.4, 's': 15},
                 height=3)
g.figure.suptitle('Pairwise RFM Feature Distributions by Cluster', y=1.02, fontsize=14)
plt.show()

## 7. Business Recommendations

Based on the cluster analysis, here are targeted strategies for each customer segment:

| Segment | Profile | Recommended Strategy |
|---------|---------|---------------------|
| **Gold - High Value** | Recent, frequent, high spenders | VIP loyalty program, early access to new products, personalized recommendations, dedicated account manager |
| **Silver - Loyal** | Moderate recency & frequency, decent spend | Upselling campaigns, cross-sell complementary categories, reward programs to push them toward Gold |
| **Bronze - Developing** | Less frequent, lower spend, but some recent activity | Engagement campaigns, discount codes, email nurturing sequences, product education content |
| **Iron - At Risk** | High recency (haven't bought recently), low frequency | Win-back campaigns, "We miss you" emails, significant discounts, exit surveys to understand churn reasons |

### Key Takeaways

1. **Pareto Principle in action:** A small percentage of customers (Gold) likely drives a disproportionate share of revenue. Protecting these relationships is critical.

2. **Re-activation potential:** The Iron/At Risk segment represents a significant portion of the customer base. Even a 10-15% reactivation rate could meaningfully impact revenue.

3. **Growth opportunity:** Silver customers are the best candidates for "moving up" -- targeted campaigns can convert them into Gold.

4. **Data-driven personalization:** Each segment should receive differentiated messaging, offers, and channel strategies based on their behavioral profile.

## 8. Conclusions

### Summary

- We generated and analyzed **~50,000 transactions** from **5,000 customers** spanning 2 years of e-commerce activity.
- **RFM Analysis** provided a structured framework to quantify customer value across three dimensions.
- **K-Means Clustering** (k=4) identified natural customer groupings that align well with business intuition.
- The **Silhouette Score** confirmed reasonable cluster separation, and the **Elbow Method** supported our choice of k.

### Next Steps

- **A/B test** marketing campaigns tailored to each segment
- Incorporate **CLV predictions** using regression models
- Add **behavioral features** (browsing patterns, cart abandonment) for richer segmentation
- Build a **real-time segmentation pipeline** that updates cluster assignments as new transactions arrive
- Explore **DBSCAN** or **Gaussian Mixture Models** for density-based alternatives

---

